# Semantic chunks on two independent Colab GPUs
Use the same notebook on two runtimes: SHARD_ID=0 and SHARD_ID=1. Both use one immutable shared plan and separate output folders. Only runtime 0 creates the plan and later merges. Select a CUDA GPU on each runtime. Read docs/SEMANTIC_STAGE5_TWO_COLABS.md before starting. All cells are supplied unexecuted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, zipfile, shutil, json
SHARD_ID = 0  # runtime A: 0; runtime B: 1
BUNDLE = Path('/content/drive/MyDrive/ViBioMIR/stage5_semantic_parallel.zip')
SHARED_ROOT = Path('/content/drive/MyDrive/ViBioMIR/semantic_parallel_run01')  # SAME physical shared folder on both runtimes
LEGACY_SEED = None  # runtime A only: Path('/content/drive/MyDrive/ViBioMIR/stage5_full_semantic') after stopping its old writer
PROJECT = Path('/content/vibiomir_semantic_parallel')
RUN_DIR = Path('/content') / f'semantic_shard_{SHARD_ID}'
BACKUP_DIR = SHARED_ROOT / f'shard_{SHARD_ID}'
PLAN = SHARED_ROOT / 'plan.json'
CPU_WORKERS = min(2, os.cpu_count() or 1)
MAX_BATCH_SIZE = 64
MAX_PADDED_TOKENS = 8192
WAVE_DOCUMENTS = 16
WAVE_SENTENCES = 1024
STOP_AFTER_BATCHES = 2  # set None after the timing/memory pilot
assert SHARD_ID in (0, 1)
assert BUNDLE.is_file(), f'Upload the shared bundle first: {BUNDLE}'
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    assert shutil.disk_usage('/content').free > sum(i.file_size for i in archive.infolist()) * 2, 'Insufficient local disk'
    for info in archive.infolist():
        assert PROJECT.resolve() in (PROJECT / info.filename).resolve().parents, 'Unsafe ZIP entry'
    archive.extractall(PROJECT)
os.chdir(PROJECT)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-stage5-semantic.txt'], check=True)
print('If Colab requests a runtime restart, restart and rerun setup before continuing.')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime on each machine'
print(torch.cuda.get_device_name(0), 'GPU GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_stage5_semantic*.py', '-v'], check=True)


In [ ]:
# Only runtime A creates the plan, ONCE. Both reuse it on every resume.
if SHARD_ID == 0 and not PLAN.is_file():
    SHARED_ROOT.mkdir(parents=True, exist_ok=True)
    seed = None
    if LEGACY_SEED is not None:
        from vibiomir.semantic_shards import snapshot_legacy_prefix
        seed = SHARED_ROOT / 'legacy_prefix'
        snapshot_legacy_prefix(LEGACY_SEED, seed)  # old writer MUST be stopped
    from vibiomir.semantic_shards import make_plan
    make_plan(PROJECT / 'inputs', PROJECT / 'tokenizer', PLAN,
        config=json.loads((PROJECT / 'configs/chunking_stage5.json').read_text()), seed_dir=seed)
assert PLAN.is_file(), 'Run the plan cell on runtime A first; then rerun this cell on runtime B.'
from vibiomir.semantic_shards import read_plan
shared_plan = read_plan(PLAN)
print('Shared plan signature (must match both machines):', shared_plan['signature'])
print('Reused old semantic batches:', shared_plan['seed_batches'])
print('Assigned [start, end) ranges:', shared_plan['ranges'], 'This machine:', SHARD_ID)


In [ ]:
command = [sys.executable, '-m', 'vibiomir.semantic_shards', 'run',
    '--input-dir', str(PROJECT / 'inputs'), '--tokenizer-dir', str(PROJECT / 'tokenizer'),
    '--plan', str(PLAN), '--shard-id', str(SHARD_ID), '--output-dir', str(RUN_DIR),
    '--backup-dir', str(BACKUP_DIR), '--cpu-workers', str(CPU_WORKERS),
    '--max-batch-size', str(MAX_BATCH_SIZE), '--max-padded-tokens', str(MAX_PADDED_TOKENS),
    '--wave-documents', str(WAVE_DOCUMENTS), '--wave-sentences', str(WAVE_SENTENCES)]
if STOP_AFTER_BATCHES is not None:
    command += ['--stop-after-batches', str(STOP_AFTER_BATCHES)]
subprocess.run(command, check=True)
state = json.loads((RUN_DIR / 'manifest.json').read_text())['state']
print('Shard state:', state)
if state != 'complete':
    print('Inspect timings and memory, set STOP_AFTER_BATCHES=None, then rerun this cell. Same shard resumes automatically.')


Run the following cell ONLY on runtime A, AFTER BOTH shard manifests say complete and both writers have stopped. It copies committed output bytes into a normal full chunk store. Merge needs CPU/disk, not a second encoder. Validation may take time because it checks every chunk's original slice and tokenizer budget.

In [ ]:
if SHARD_ID == 0:
    from vibiomir.semantic_shards import merge_shards
    from vibiomir.chunk_pipeline import validate_chunks
    from vibiomir.chunk_models import load_tokenizer
    from vibiomir.chunk_colab import save_checkpoint, restore_checkpoint
    for i in (0, 1):
        path = SHARED_ROOT / f'shard_{i}/manifest.json'
        assert path.is_file() and json.loads(path.read_text())['state'] == 'complete', f'Shard {i} is not finished yet'
    MERGED = Path('/content/semantic_merged')
    FINAL_BACKUP = SHARED_ROOT / 'merged'
    restore_checkpoint(FINAL_BACKUP, MERGED)
    prefix = SHARED_ROOT / 'legacy_prefix' if shared_plan['seed_batches'] else None
    merge_shards(PROJECT / 'inputs', PLAN, [SHARED_ROOT / 'shard_0', SHARED_ROOT / 'shard_1'], MERGED, seed_dir=prefix)
    save_checkpoint(MERGED, FINAL_BACKUP)
    tokenizer, signature = load_tokenizer(PROJECT / 'tokenizer')
    print(validate_chunks(PROJECT / 'inputs', MERGED, tokenizer, signature))
    save_checkpoint(MERGED, FINAL_BACKUP)
    print('Full validated output:', FINAL_BACKUP)
else:
    print('Merge is performed only by runtime A after both shards finish.')
